# Visualization render-path check

A minimal probe: the same tiny graph rendered twice — once as standalone HTML
(`render()`), once as an interactive widget (`render_widget()`). Run all cells,
then report which of A / B shows nodes and edges (and which one, if any, keeps
growing the cell's height without painting).

This is for validating the Relate notebook visualizations inside **VS Code**,
whose webview host treats the two paths differently.

In [ ]:
import subprocess
import sys

def _ensure(module, pip_spec):
    try:
        __import__(module)
    except ImportError:
        print(f"Installing {pip_spec} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pip_spec.split()])

_ensure("relate", "../surfaces/relate-python")  # local wheel (pre-PyPI)
# _ensure("relate", "relate-db")  # published-wheel placeholder
# 1.12: no [notebook] extra (Jupyter deps are core); needs Python >= 3.11.
_ensure("neo4j_viz", "neo4j-viz>=1.12.0")

import relate
from relate.viz import to_visualization_graph
print("relate", getattr(relate, "__version__", "local build"))

try:
    import anywidget
    print("anywidget", anywidget.__version__)
except ImportError:
    print("anywidget NOT installed — render_widget() cannot work here")

## A tiny graph, built once

In [ ]:
db = relate.connect(":memory:")
db.execute("""
    CREATE (a:Tag {name: 'neo4j'})
    CREATE (b:Tag {name: 'cypher'})
    CREATE (q:Question {title: 'How do I...?'})-[:TAGGED]->(a)-[:SIMILAR_TO {jaccard: 0.75}]->(b)
    """)

vg = to_visualization_graph(
    db.execute("MATCH (a)-[r]-(b) RETURN a, r, b")
)
print(f"{len(vg.nodes)} nodes, {len(vg.relationships)} relationships")

## A — `render()` (standalone HTML)
The original approach: one self-contained HTML document in the output.

In [ ]:
vg.render(initial_zoom=2)

## B — `render_widget()` (interactive anywidget)
The path neo4j-viz documents for Jupyter/VS Code/Colab.

In [ ]:
vg.render_widget(initial_zoom=2)

---
Both should show 3 nodes and 2 relationships (the TAGGED and SIMILAR_TO edges),
colored by label. Report what each cell does — painted, blank, or a canvas that
keeps growing in height.